<!-- CURRICULUM_HEADER_START -->
<div class="mh"><p class="mh-crumb"><a href="/series/earth-data/index.html">Earth Data in Python and R</a><span class="sep">·</span>Part 6 · Maps</p><p class="mh-facts"><span class="lvl lvl-introductory">Introductory</span></p><p class="mh-assumes">Assumes <span class="unwritten" title="not written yet">GeoPandas basics</span> and <span class="unwritten" title="not written yet">Choropleth pitfalls</span>.</p></div>
<!-- CURRICULUM_HEADER_END -->

::: {.callout-note appearance="simple" icon=false}
## TL;DR

A county table of PM2.5 concentrations and a Census shapefile share a five-digit FIPS code and nothing else. This module joins the two in GeoPandas, draws an interactive choropleth with `explore()`, and then cuts the GeoJSON embedded in the page from 32 MB to a size a static host will serve. You leave able to build a county map from any FIPS-keyed table and to keep the result publishable.
:::

The first cell installs the stack (`pandas`, `geopandas`, `folium`, `matplotlib`, `mapclassify`); the last three are what `explore()` needs to draw and classify the colour scale.

In [1]:
# %pip install pandas geopandas folium matplotlib mapclassify

In [2]:
import pandas as pd
import geopandas as gpd

## The county table

The source is a CSV export from the NIH HDPulse data portal (link in the cell below): the average daily PM2.5 concentration, in micrograms per cubic metre (µg/m³), for each US county. The export has a five-line preamble, a national total in the first row and footnotes after the last county, so the read and clean-up cells do three things. They skip the preamble with `skiprows=5`. They coerce the last column to a float with `errors="coerce"` and drop rows where either FIPS or the reading is missing, which removes the footnotes. And they zero-pad FIPS back to five characters: `read_csv` parses it as a number, which strips the leading zero from every state coded 01 to 09, and without the padding Alabama through Connecticut would silently fail the join.

In [3]:
# download the data from US NIH (https://hdpulse.nimhd.nih.gov/data-portal/physical/table?age=001&age_options=ageall_1&demo=234&demo_options=air_pollution_1&physicaltopic=002&physicaltopic_options=physical_2&race=00&race_options=raceall_1&sex=0&sex_options=sexboth_1&statefips=99&statefips_options=area_states)

county_pm25: pd.DataFrame = pd.read_csv(
    filepath_or_buffer="HDPulse_data_export.csv",
    skiprows=5,
)

In [4]:
county_pm25

,County,FIPS,Micrograms per cubic meter (PM2.5)(2)
0,United States,0.0,7.4
1,"San Bernardino County, California",6071.0,15.6
2,"Fairbanks North Star, Alaska",2090.0,15.5
3,"Allegheny County, Pennsylvania",42003.0,14.1
4,"San Diego County, California",6073.0,13.8
...,...,...,...
3146,Notes:,NaN,NaN
3147,Source: National Environmental Public Health T...,NaN,NaN
3148,Average daily density of fine particulate matt...,NaN,NaN
3149,Some data are not available or suppressed due ...,NaN,NaN


In [5]:
county_pm25_processed: pd.DataFrame = (
    county_pm25.assign(
        # make PM2.5 reading a float
        pm25_ug_per_m3=lambda x: pd.to_numeric(arg=x[x.keys()[-1]], errors="coerce"),
        # convert FIPS to a 5-digit string
        FIPS=lambda x: pd.to_numeric(x["FIPS"]),
    )
    .dropna(
        # drop rows with missing PM2.5 readings
        subset=[
            "FIPS",
            "pm25_ug_per_m3",
        ],
    )
    .assign(
        FIPS=lambda x: x["FIPS"].astype(int).astype(str).str.zfill(5),
    )
)

In [6]:
# optional sense check
county_pm25_processed

,County,FIPS,Micrograms per cubic meter (PM2.5)(2),pm25_ug_per_m3
0,United States,00000,7.4,7.4
1,"San Bernardino County, California",06071,15.6,15.6
2,"Fairbanks North Star, Alaska",02090,15.5,15.5
3,"Allegheny County, Pennsylvania",42003,14.1,14.1
4,"San Diego County, California",06073,13.8,13.8
...,...,...,...,...
3111,"Custer County, South Dakota",46033,2.6,2.6
3112,"Apache County, Arizona",04001,2.5,2.5
3113,"Campbell County, Wyoming",56005,2.4,2.4
3114,"Converse County, Wyoming",56009,2.2,2.2


## County geometry from the Census

The Census Bureau's cartographic boundary file at 1:500,000 (`cb_2017_us_county_500k`) carries state and county FIPS as separate strings, `STATEFP` and `COUNTYFP`. Concatenating them gives the same five-character key the PM2.5 table now has. The merge runs from the geometry side with `how="left"`, so every county keeps its outline; a county without a reading stays on the map as a gap rather than disappearing, and the national-total row (FIPS `00000`) falls away because no county carries it.

In [7]:
# download us county shape files from https://www.census.gov/geographies/mapping-files/time-series/geo/carto-boundary-file.html

counties: gpd.GeoDataFrame = gpd.read_file(
    filename="cb_2017_us_county_500k",
)

In [8]:
counties_processed: gpd.GeoDataFrame = counties.assign(
    FIPS=lambda x: x["STATEFP"] + x["COUNTYFP"],
)

In [9]:
# optional sense check
counties_processed

,STATEFP,COUNTYFP,COUNTYNS,AFFGEOID,GEOID,NAME,LSAD,ALAND,AWATER,AgriRegion,geometry,FIPS
0,01,005,00161528,0500000US01005,01005,Barbour,06,2292144656,50538698,EastUS,"POLYGON ((-85.74803 31.61918, -85.74544 31.618...",01005
1,01,023,00161537,0500000US01023,01023,Choctaw,06,2365869837,19144469,EastUS,"POLYGON ((-88.47323 31.89386, -88.46888 31.930...",01023
2,01,035,00161543,0500000US01035,01035,Conecuh,06,2201948618,6643480,EastUS,"POLYGON ((-87.4272 31.26436, -87.42551 31.2683...",01035
3,01,051,00161551,0500000US01051,01051,Elmore,06,1601762124,99965171,EastUS,"POLYGON ((-86.41333 32.75059, -86.37115 32.750...",01051
4,01,065,00161558,0500000US01065,01065,Hale,06,1667907107,32423356,EastUS,"POLYGON ((-87.87046 32.76244, -87.86818 32.765...",01065
...,...,...,...,...,...,...,...,...,...,...,...,...
3228,37,069,01008553,0500000US37069,37069,Franklin,06,1273631713,7304032,EastUS,"POLYGON ((-78.54551 36.0567, -78.54493 36.0772...",37069
3229,48,317,01383941,0500000US48317,48317,Martin,06,2369724595,1931832,CentralRegion,"POLYGON ((-102.21103 32.17704, -102.21111 32.3...",48317
3230,54,107,01560558,0500000US54107,54107,Wood,06,948592039,27228519,EastUS,"POLYGON ((-81.75582 39.18052, -81.75575 39.180...",54107
3231,13,269,00344156,0500000US13269,13269,Taylor,06,975612265,7802363,EastUS,"MULTIPOLYGON (((-84.05331 32.52202, -84.00849 ...",13269


In [10]:
# merge the two dataframes
counties_w_pm25: gpd.GeoDataFrame = counties_processed.merge(
    right=county_pm25_processed,
    on="FIPS",
    how="left",
)

In [11]:
# optional sense check
counties_w_pm25

,STATEFP,COUNTYFP,COUNTYNS,AFFGEOID,GEOID,NAME,LSAD,ALAND,AWATER,AgriRegion,geometry,FIPS,County,Micrograms per cubic meter (PM2.5)(2),pm25_ug_per_m3
0,01,005,00161528,0500000US01005,01005,Barbour,06,2292144656,50538698,EastUS,"POLYGON ((-85.74803 31.61918, -85.74544 31.618...",01005,"Barbour County, Alabama",9.4,9.4
1,01,023,00161537,0500000US01023,01023,Choctaw,06,2365869837,19144469,EastUS,"POLYGON ((-88.47323 31.89386, -88.46888 31.930...",01023,"Choctaw County, Alabama",9.3,9.3
2,01,035,00161543,0500000US01035,01035,Conecuh,06,2201948618,6643480,EastUS,"POLYGON ((-87.4272 31.26436, -87.42551 31.2683...",01035,"Conecuh County, Alabama",9.2,9.2
3,01,051,00161551,0500000US01051,01051,Elmore,06,1601762124,99965171,EastUS,"POLYGON ((-86.41333 32.75059, -86.37115 32.750...",01051,"Elmore County, Alabama",10.0,10.0
4,01,065,00161558,0500000US01065,01065,Hale,06,1667907107,32423356,EastUS,"POLYGON ((-87.87046 32.76244, -87.86818 32.765...",01065,"Hale County, Alabama",9.4,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3228,37,069,01008553,0500000US37069,37069,Franklin,06,1273631713,7304032,EastUS,"POLYGON ((-78.54551 36.0567, -78.54493 36.0772...",37069,"Franklin County, North Carolina",8.5,8.5
3229,48,317,01383941,0500000US48317,48317,Martin,06,2369724595,1931832,CentralRegion,"POLYGON ((-102.21103 32.17704, -102.21111 32.3...",48317,"Martin County, Texas",7.5,7.5
3230,54,107,01560558,0500000US54107,54107,Wood,06,948592039,27228519,EastUS,"POLYGON ((-81.75582 39.18052, -81.75575 39.180...",54107,"Wood County, West Virginia",7.8,7.8
3231,13,269,00344156,0500000US13269,13269,Taylor,06,975612265,7802363,EastUS,"MULTIPOLYGON (((-84.05331 32.52202, -84.00849 ...",13269,"Taylor County, Georgia",9.6,9.6


## The map

`GeoDataFrame.explore()` returns a folium map with a colour scale on `pm25_ug_per_m3`, a hover tooltip and a click popup. Everything the map needs is embedded in the page as GeoJSON, which is where the size problem comes from: the full-fidelity version is 32 MB, past the 25 MiB per-asset limit of the host. The cell below keeps the problem in check three ways. It selects only the four columns the map uses, because `tooltip` controls what is displayed, not what is embedded. It simplifies the outlines with a 0.005 degree tolerance, roughly 500 m, which is invisible at national zoom and keeps 13% of the 1.04 million vertices without dropping a county. And it names the tooltip and popup fields explicitly rather than letting folium serialise every column.

In [12]:
# Display the data on a map, coloured by PM2.5 level.
#
# Three things keep the rendered page publishable — the full-fidelity
# version is 32 MB, past the 25 MiB per-asset limit our host allows:
#
# 1. Keep only the columns the map uses. `tooltip` controls what is
#    *displayed*, not what is *embedded* — without this, every county
#    ships its FIPS codes, land and water areas, and so on.
# 2. `simplify()` — full-resolution outlines carry ~1.04M coordinates.
#    At national zoom ~500 m of boundary detail is invisible, and a 0.005
#    degree tolerance keeps 13% of the vertices while dropping no counties.
# 3. Name the tooltip and popup fields explicitly.
counties_w_pm25[["NAME", "County", "pm25_ug_per_m3", "geometry"]].assign(
    geometry=lambda x: x.geometry.simplify(tolerance=0.005),
).explore(
    column="pm25_ug_per_m3",
    tooltip=["NAME", "pm25_ug_per_m3"],
    popup=["NAME", "County", "pm25_ug_per_m3"],
)

## Key takeaways

- FIPS is the only key the health table and the Census geometry share; make it a zero-padded five-character string on both sides or the join silently drops the states coded 01 to 09.
- Coerce the measurement column with `errors="coerce"` and drop the resulting `NaN` rows; footnotes and preamble leftovers go in one step.
- Merge from the geometry side with `how="left"` so counties without data stay visible as gaps rather than vanishing.
- `explore()` embeds the whole GeoDataFrame in the page; select columns and `simplify()` the geometry before calling it, or the map will not fit on a static host.

<!-- CURRICULUM_FOOTER_START -->
<div class="mf"><a class="mf-prev" href="/blogs/ms-buildings/index.html"><span>Previous</span>Downloading building footprints</a><a class="mf-up" href="/series/earth-data/index.html"><span>Series</span>Earth Data in Python and R</a><a class="mf-next" href="/blogs/atlantic-centered-map/index.html"><span>Next</span>An Atlantic-centred map</a></div>
<!-- CURRICULUM_FOOTER_END -->